In [11]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [12]:
df = pd.read_csv("cleaned_data1.csv")

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11123 entries, 0 to 11122
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   bookID              11123 non-null  int64  
 1   title               11123 non-null  object 
 2   authors             11123 non-null  object 
 3   average_rating      11123 non-null  float64
 4   isbn                11123 non-null  object 
 5   isbn13              11123 non-null  int64  
 6   language_code       11123 non-null  object 
 7     num_pages         11123 non-null  int64  
 8   ratings_count       11123 non-null  int64  
 9   text_reviews_count  11123 non-null  int64  
 10  publication_date    11123 non-null  object 
 11  publisher           11123 non-null  object 
 12  tags                11123 non-null  object 
dtypes: float64(1), int64(5), object(7)
memory usage: 1.1+ MB


In [14]:
print(df.columns)

Index(['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13',
       'language_code', '  num_pages', 'ratings_count', 'text_reviews_count',
       'publication_date', 'publisher', 'tags'],
      dtype='object')


In [15]:
print(df.shape)
df.head()

(11123, 13)


,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher,tags
0,1,Harry-Potter-and-the-Half-Blood-Prince-(Harry-...,J.K.-Rowling/Mary-GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic-Inc.,1 harry-potter-and-the-half-blood-prince-(harr...
1,2,Harry-Potter-and-the-Order-of-the-Phoenix-(Har...,J.K.-Rowling/Mary-GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic-Inc.,2 harry-potter-and-the-order-of-the-phoenix-(h...
2,4,Harry-Potter-and-the-Chamber-of-Secrets-(Harry...,J.K.-Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic,4 harry-potter-and-the-chamber-of-secrets-(har...
3,5,Harry-Potter-and-the-Prisoner-of-Azkaban-(Harr...,J.K.-Rowling/Mary-GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic-Inc.,5 harry-potter-and-the-prisoner-of-azkaban-(ha...
4,8,Harry-Potter-Boxed-Set--Books-1-5-(Harry-Potte...,J.K.-Rowling/Mary-GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic,8 harry-potter-boxed-set--books-1-5-(harry-pot...


In [16]:
df['tags'] = df['title'].fillna('') + ' ' + df['authors'].fillna('')

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

tv = TfidfVectorizer(max_features=10000, stop_words='english')
vector = tv.fit_transform(df['tags']).toarray()

print(vector.shape)

(11123, 10000)


In [18]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vector)
similarity

array([[1.        , 0.76774559, 0.66386514, ..., 0.        , 0.        ,
        0.        ],
       [0.76774559, 1.        , 0.67580157, ..., 0.        , 0.        ,
        0.        ],
       [0.66386514, 0.67580157, 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.40028501,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.40028501, 1.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        1.        ]], shape=(11123, 11123))

In [19]:
def get_name_by_index(i):
    
    if i < len(df) and i>0:
        return df.loc[i,'title']
    else:
        return 'Index Not found'

In [20]:
a = get_name_by_index(5)
a

'Unauthorized-Harry-Potter-Book-Seven-News:-"Half-Blood-Prince"-Analysis-and-Speculation'

In [21]:
def get_index_from_name(name):
    clean_user_name = name.strip().lower().replace(' ', '').replace('-', '')
    match = df[df['title'].str.lower().str.replace(' ', '').str.replace('-', '') == clean_user_name]
    
    if not match.empty:
        return match.index[0]
    return -1

In [22]:
b = get_index_from_name("spider man 3")
b

-1

In [24]:
name = input("Enter book you readed : ")
index = get_index_from_name(name)

if index != -1:
    
    similarity_indexes = list(enumerate(similarity[index]))

    similarity_indexes = sorted(similarity_indexes, key=lambda x: x[1], reverse=True)
    
    print(f"\nBecause you readed '{df.loc[index, 'title']}':")
    print("You must watch the following books:")
    
    for i in range(1, 6):
        movie_idx = similarity_indexes[i][0]
        print(i, ":", get_name_by_index(movie_idx))
else:
    print("books Not Found")

Enter book you readed :  Harry Potter and the Half-Blood Prince (Harry Potter  #6)



Because you readed 'Harry-Potter-and-the-Half-Blood-Prince-(Harry-Potter--#6)':
You must watch the following books:
1 : Harry-Potter-and-the-Half-Blood-Prince-(Harry-Potter--#6)
2 : Harry-Potter-and-the-Sorcerer's-Stone-(Harry-Potter--#1)
3 : Harry-Potter-and-the-Goblet-of-Fire-(Harry-Potter--#4)
4 : Harry-Potter-and-the-Chamber-of-Secrets-(Harry-Potter--#2)
5 : Harry-Potter-and-the-Order-of-the-Phoenix-(Harry-Potter--#5)


In [25]:
import pickle as pkl
similarity=similarity.astype("float32")
top_similar=[]
for i in range(len(similarity)):
    indices=np.argsort(similarity[i])[-21:-1][::-1]
    source = similarity[i][indices]
    top_similar.append(list(zip(indices,source)))
pkl.dump(top_similar, open('similarity.pkl', 'wb'))